# LTL satisfiability across the MTSA branches

Runs the `tools/ltl-sat` checker (`LTLSatService`: model check the **universal
LTS** over the formula's alphabet against the formula, or against its negation,
and answer SAT/UNSAT) on the same set of formulas on each branch that ships the
tool, and records, per formula and per branch:

* the verdict (`SAT` / `UNSAT`, or `ERROR` when the branch cannot even parse the
  formula — that is how `masterTests` reacts to the past operators),
* the wall-clock time of the check,
* the Java-side split of that time (property compilation vs. model check) and
  the size of the automaton that was checked.

Each branch is run from its own `git worktree`, so the working tree you are
sitting in is never touched. The first run per branch triggers a `mvn clean
compile` in that worktree (minutes); later runs reuse it.

This notebook is deliberately **not committed** — it lives only in this working
tree.

## 1. Configuration

In [1]:
import json
import os
import shutil
import statistics
import subprocess
import sys
import time
from dataclasses import dataclass, field, asdict

#: branches to compare; each one must carry tools/ltl-sat
BRANCHES = ["masterTests", "pLTLTests", "pLTLpdTests"]

#: where the throw-away worktrees for the other branches go
WORKTREE_ROOT = "/tmp/mtsa-ltlsat"

#: "sat" -> SAT/UNSAT, "valid" -> VALID/INVALID
MODE = "sat"

#: None = derive the universal LTS alphabet from each formula's propositions;
#: a list like ["a", "b", "c"] forces one common alphabet for every formula
ALPHABET = None

#: give up on a single formula after this many seconds (the service is then
#: restarted, the remaining formulas still run)
TIMEOUT = 120.0

#: how many times to check each formula; the reported time is the minimum
REPEATS = 1

#: file with one formula per line (see section 4). It is created from the
#: built-in set the first time, so point it at your own file to use that
#: instead. Set to None to skip the file and use the built-in list.
FORMULA_FILE = "formulas.txt"

#: where the results are written
CSV_RESULTS = "ltl_sat_results.csv"              # one row per (formula, branch)
CSV_BY_FORMULA = "ltl_sat_results_by_formula.csv"  # one row per formula, a column per branch

REPO_ROOT = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    cwd=os.getcwd(), capture_output=True, text=True, check=True,
).stdout.strip()
PROJECT_SUBDIR = os.path.join("maven-root", "mtsa")

CURRENT_BRANCH = subprocess.run(
    ["git", "-C", REPO_ROOT, "rev-parse", "--abbrev-ref", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()

#: relative paths above are resolved against the notebook's directory
HERE = os.path.abspath(os.getcwd())
resolve = lambda p: None if p is None else os.path.join(HERE, p) if not os.path.isabs(p) else p
FORMULA_FILE = resolve(FORMULA_FILE)
CSV_RESULTS = resolve(CSV_RESULTS)
CSV_BY_FORMULA = resolve(CSV_BY_FORMULA)

print("repo           :", REPO_ROOT)
print("current branch :", CURRENT_BRANCH)
print("branches       :", ", ".join(BRANCHES))
print("formula file   :", FORMULA_FILE)
print("results        :", CSV_RESULTS)

repo           : /home/lakso/Programas/mtsa
current branch : pLTLpdTests
branches       : masterTests, pLTLTests, pLTLpdTests
formula file   : /home/lakso/Programas/mtsa/maven-root/mtsa/tools/ltl-sat/formulas.txt
results        : /home/lakso/Programas/mtsa/maven-root/mtsa/tools/ltl-sat/ltl_sat_results.csv


## 2. One working tree per branch

The branch this notebook lives on is measured in place (it is already checked
out); every other branch gets a worktree under `WORKTREE_ROOT`. Nothing is ever
checked out over your current tree.

In [2]:
def project_dir(branch):
    """The maven-root/mtsa directory to run `branch` from."""
    if branch == CURRENT_BRANCH:
        return os.path.join(REPO_ROOT, PROJECT_SUBDIR)
    return os.path.join(WORKTREE_ROOT, branch, PROJECT_SUBDIR)


def ensure_worktree(branch):
    """Create the worktree for `branch` if it is not there yet."""
    if branch == CURRENT_BRANCH:
        return project_dir(branch)
    path = os.path.join(WORKTREE_ROOT, branch)
    if not os.path.isdir(os.path.join(path, ".git")) and not os.path.isfile(
            os.path.join(path, ".git")):
        os.makedirs(WORKTREE_ROOT, exist_ok=True)
        subprocess.run(["git", "-C", REPO_ROOT, "worktree", "add", path, branch],
                       check=True)
    else:
        # keep an existing worktree on the tip of its branch
        subprocess.run(["git", "-C", path, "checkout", branch], check=True,
                       capture_output=True, text=True)
    return project_dir(branch)


def build(branch, quiet=True):
    """Compile the branch (no-op when the launcher's cache is already warm)."""
    directory = ensure_worktree(branch)
    script = os.path.join(directory, "tools", "ltl-sat", "ltl-sat-server.sh")
    if not os.path.isfile(script):
        raise RuntimeError(f"{branch} does not ship tools/ltl-sat ({script})")
    started = time.perf_counter()
    # running the launcher with no requests builds, starts, and exits
    subprocess.run(["bash", script], cwd=directory, stdin=subprocess.DEVNULL,
                   stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL if quiet else None, check=True)
    return time.perf_counter() - started


for branch in BRANCHES:
    seconds = build(branch)
    head = subprocess.run(["git", "-C", project_dir(branch), "rev-parse", "--short", "HEAD"],
                          capture_output=True, text=True, check=True).stdout.strip()
    print(f"{branch:<14} {head}  ready in {seconds:6.1f}s  ({project_dir(branch)})")

masterTests    0331a27f0  ready in    0.3s  (/tmp/mtsa-ltlsat/masterTests/maven-root/mtsa)


pLTLTests      099d8381c  ready in    0.4s  (/tmp/mtsa-ltlsat/pLTLTests/maven-root/mtsa)


pLTLpdTests    c67c8bc73  ready in    0.4s  (/home/lakso/Programas/mtsa/maven-root/mtsa)


## 3. The client

Each branch carries its own copy of `ltl_sat_client.py`; they are identical, so
the one from the current branch is imported once and used for every service.

In [3]:
client_dir = os.path.join(REPO_ROOT, PROJECT_SUBDIR, "tools", "ltl-sat")
if client_dir not in sys.path:
    sys.path.insert(0, client_dir)

from ltl_sat_client import LTLSat, LTLSatError, LTLSatTimeout  # noqa: E402

print("client:", os.path.join(client_dir, "ltl_sat_client.py"))

client: /home/lakso/Programas/mtsa/maven-root/mtsa/tools/ltl-sat/ltl_sat_client.py


## 4. The formulas

The set below is the built-in default. It is written to `FORMULA_FILE` the first
time this notebook runs, and from then on **the file is what gets checked** — so
drop your own file there (or point `FORMULA_FILE` at it) and re-run.

File format: one formula per line; blank lines are ignored, `#` lines are
comments, `# group: <name>` labels the formulas that follow, and a trailing
`# SAT` / `# UNSAT` / `# expect=VALID` on a formula line records the expected
verdict. (A formula line must therefore not contain a whitespace-preceded `#`.)

The `group` and `expected` columns are only labels for the report. `future`
formulas work on every branch; `past` ones use `<*>` (once), `[*]`
(historically), `Y` (previous) and `S` (since), which only the pLTL branches
parse.

Remember that MTSA evaluates an assertion **at time 0**, so at the root a past
operator collapses: `<*>a` is `a` there, which is why `<*>a && !a` is `UNSAT`.
Also, the verdict is relative to the universal LTS alphabet: with the default
`ALPHABET = None` that alphabet is exactly the formula's propositions, so e.g.
`<>!a` is `UNSAT` (the only trace over `{a}` is `a,a,a,...`) while it is `SAT`
over `{a,b}`.

In [4]:
FORMULAS = [
    # (group, formula, expected verdict or None)
    ("propositional", "a",                                   "SAT"),
    ("propositional", "a && !a",                             "UNSAT"),
    ("propositional", "a || !a",                             "SAT"),
    ("tautology",     "a -> a",                                "SAT"),
    ("tautology",     "[]a -> a",                              "SAT"),
    ("tautology",     "[](a -> b) -> ([]a -> []b)",            "SAT"),
    ("contradiction", "!(a -> a)",                             "UNSAT"),
    ("future",        "[]a",                                 "SAT"),
    ("future",        "<>a",                                 "SAT"),
    ("future",        "<>!a",                                "UNSAT"),   # alphabet is {a}
    ("future",        "[]<>a",                               "SAT"),
    ("future",        "<>[]a",                               "SAT"),
    ("future",        "X X X a",                             "SAT"),
    ("future",        "a U b",                               "SAT"),
    ("future",        "a W b",                               "SAT"),
    ("future",        "[](a -> <>b)",                        "SAT"),
    ("future",        "[](a -> X b)",                        "SAT"),
    ("future",        "[]a && <>b",                          "UNSAT"),
    ("future",        "(a U b) && []!b",                     "UNSAT"),
    ("future",        "[](a -> <>b) && []<>a && []!b",       "UNSAT"),
    ("future",        "[](a -> X(b U c)) && []<>a && []!c",  "UNSAT"),
    ("future",        "<>(a && X b) && [](b -> X c)",        "SAT"),
    ("future",        "[]<>(a -> X(b U c))",                 "SAT"),
    ("future big",    "(((a U b) U c) U d)",                 "SAT"),
    ("future big",    "[]<>a && []<>b && []<>c",             "SAT"),
    ("future big",    "[]<>a && []<>b && []<>c && []<>d",    "SAT"),
    ("past",          "<*>a",                                "SAT"),
    ("past",          "[*]a",                                "SAT"),
    ("past",          "Y a",                                 "UNSAT"),   # nothing before time 0
    ("past",          "Y a -> <*>a",                         "SAT"),
    ("past",          "a S b",                               "SAT"),
    ("past",          "<*>a && !a",                          "UNSAT"),   # <*>a is a at time 0
    ("past",          "[*]!a && <*>a",                       "UNSAT"),
    ("past",          "<*>(a && Y b)",                       "UNSAT"),
    ("past",          "Y Y Y Y a",                           "UNSAT"),
    ("past",          "<*><*><*>a",                          "SAT"),
    ("past",          "[](b -> <*>a)",                       "SAT"),
    ("past",          "[](b -> Y a)",                        "SAT"),
    ("past",          "[](b -> <*>a) && b",                  "UNSAT"),
    ("past",          "[](b -> (a S c))",                    "SAT"),
    ("past big",      "[](a -> <*>(b && Y c))",              "SAT"),
    ("past big",      "[]<>a && [](a -> <*>b)",              "SAT"),
    ("past big",      "[]<>a && []<>b && [](a -> <*>b) && [](b -> <*>c)", "SAT"),
]

BUILTIN_FORMULAS = FORMULAS


def save_formula_file(path, formulas=BUILTIN_FORMULAS):
    """Write `formulas` as a one-formula-per-line file."""
    with open(path, "w") as fh:
        print("# one LTL formula per line; '#' starts a comment", file=fh)
        print("# '# group: <name>' labels the formulas that follow", file=fh)
        print("# a trailing '# SAT' / '# UNSAT' records the expected verdict", file=fh)
        group = None
        for g, formula, expected in formulas:
            if g != group:
                print(file=fh)
                print(f"# group: {g}", file=fh)
                group = g
            print(formula + (f"    # {expected}" if expected else ""), file=fh)
    return path


def load_formula_file(path):
    """Read a one-formula-per-line file into [(group, formula, expected)]."""
    verdicts = ("SAT", "UNSAT", "VALID", "INVALID")
    formulas, group = [], "file"
    with open(path) as fh:
        for raw in fh:
            line = raw.strip()
            if not line:
                continue
            if line.startswith("#"):
                comment = line.lstrip("#").strip()
                if comment.lower().startswith("group"):
                    group = comment.split(":", 1)[-1].split("=", 1)[-1].strip() or group
                continue
            expected = None
            head, sep, comment = line.partition(" #")       # trailing annotation
            if sep:
                line = head.strip()
                word = comment.strip().split("=")[-1].split()[0].upper() if comment.strip() else ""
                expected = word if word in verdicts else None
            formulas.append((group, line, expected))
    return formulas


if FORMULA_FILE:
    if not os.path.exists(FORMULA_FILE):
        save_formula_file(FORMULA_FILE)
        print("created", FORMULA_FILE, "from the built-in set")
    FORMULAS = load_formula_file(FORMULA_FILE)
    print("read", len(FORMULAS), "formulas from", FORMULA_FILE)

print(len(FORMULAS), "formulas,", len(BRANCHES), "branches ->",
      len(FORMULAS) * len(BRANCHES) * REPEATS, "checks")

created /home/lakso/Programas/mtsa/maven-root/mtsa/tools/ltl-sat/formulas.txt from the built-in set
read 43 formulas from /home/lakso/Programas/mtsa/maven-root/mtsa/tools/ltl-sat/formulas.txt
43 formulas, 3 branches -> 129 checks


## 5. Run every formula on every branch

In [5]:
@dataclass
class Row:
    branch: str
    group: str
    formula: str
    expected: str = None
    verdict: str = None          # SAT / UNSAT / VALID / INVALID / ERROR / TIMEOUT
    seconds: float = None        # best wall-clock time of REPEATS runs
    compile_ms: int = None       # Java: building the property automaton
    check_ms: int = None         # Java: model checking the universal LTS
    buchi_states: int = None
    buchi_transitions: int = None
    is_property: bool = None     # the formula compiled to a safety property
    alphabet: str = None
    detail: str = None           # error message / counterexample
    times: list = field(default_factory=list)


def run_branch(branch, formulas=FORMULAS, mode=MODE, alphabet=ALPHABET,
               timeout=TIMEOUT, repeats=REPEATS, verbose=True):
    """Check every formula on one branch, returning a list of Row."""
    rows = []
    svc = LTLSat.spawn(project_dir(branch), quiet=True)
    try:
        for group, formula, expected in formulas:
            row = Row(branch=branch, group=group, formula=formula, expected=expected)
            for _ in range(repeats):
                try:
                    res = svc.check(formula, mode=mode, alphabet=alphabet,
                                    timeout=timeout, raise_on_error=False)
                except LTLSatTimeout:
                    row.verdict = "TIMEOUT"
                    row.detail = f"no answer in {timeout}s"
                    svc = svc.restart()
                    break
                except LTLSatError as e:          # the service died on us
                    row.verdict = "ERROR"
                    row.detail = str(e)
                    svc = svc.restart()
                    break
                row.times.append(res.wall_seconds)
                if res.ok:
                    row.verdict = res.result
                    row.compile_ms = res.compile_millis
                    row.check_ms = res.check_millis
                    row.buchi_states = res.buchi_states
                    row.buchi_transitions = res.buchi_transitions
                    row.is_property = res.is_property
                    row.alphabet = ",".join(res.alphabet)
                    row.detail = " ".join(res.trace)
                else:
                    row.verdict = "ERROR"
                    row.detail = res.error
                    break
            if row.times:
                row.seconds = min(row.times)
            rows.append(row)
            if verbose:
                mark = ""
                if row.expected and row.verdict not in (row.expected, None):
                    mark = f"  <- expected {row.expected}"
                secs = f"{row.seconds:7.3f}s" if row.seconds is not None else "      --"
                print(f"  {secs}  {row.verdict:<7} {row.formula}{mark}")
    finally:
        svc.close()
    return rows


rows = []
for branch in BRANCHES:
    print(f"=== {branch}")
    started = time.perf_counter()
    rows += run_branch(branch)
    print(f"--- {branch} done in {time.perf_counter() - started:.1f}s")

print(len(rows), "measurements")

=== masterTests


    0.582s  SAT     a
    0.014s  UNSAT   a && !a
    0.023s  UNSAT   a || !a  <- expected SAT
    0.018s  UNSAT   a -> a  <- expected SAT
    0.032s  SAT     []a -> a
    0.033s  SAT     [](a -> b) -> ([]a -> []b)
    0.025s  UNSAT   !(a -> a)
    0.014s  SAT     []a
    0.030s  SAT     <>a


    0.047s  UNSAT   <>!a
    0.013s  SAT     []<>a
    0.028s  SAT     <>[]a
    0.033s  SAT     X X X a
    0.033s  SAT     a U b
    0.015s  SAT     a W b
    0.039s  SAT     [](a -> <>b)
    0.025s  SAT     [](a -> X b)


    0.028s  UNSAT   []a && <>b
    0.023s  UNSAT   (a U b) && []!b
    0.023s  UNSAT   [](a -> <>b) && []<>a && []!b
    0.029s  UNSAT   [](a -> X(b U c)) && []<>a && []!c
    0.043s  SAT     <>(a && X b) && [](b -> X c)
    0.038s  SAT     []<>(a -> X(b U c))


    0.063s  SAT     (((a U b) U c) U d)
    0.021s  SAT     []<>a && []<>b && []<>c
    0.028s  SAT     []<>a && []<>b && []<>c && []<>d
    0.002s  ERROR   <*>a  <- expected SAT
    0.012s  ERROR   [*]a  <- expected SAT
    0.003s  ERROR   Y a  <- expected UNSAT
    0.002s  ERROR   Y a -> <*>a  <- expected SAT
    0.006s  ERROR   a S b  <- expected SAT
    0.001s  ERROR   <*>a && !a  <- expected UNSAT
    0.005s  ERROR   [*]!a && <*>a  <- expected UNSAT
    0.008s  ERROR   <*>(a && Y b)  <- expected UNSAT
    0.002s  ERROR   Y Y Y Y a  <- expected UNSAT
    0.001s  ERROR   <*><*><*>a  <- expected SAT
    0.001s  ERROR   [](b -> <*>a)  <- expected SAT
    0.002s  ERROR   [](b -> Y a)  <- expected SAT
    0.000s  ERROR   [](b -> <*>a) && b  <- expected UNSAT
    0.000s  ERROR   [](b -> (a S c))  <- expected SAT
    0.000s  ERROR   [](a -> <*>(b && Y c))  <- expected SAT
    0.000s  ERROR   []<>a && [](a -> <*>b)  <- expected SAT
    0.000s  ERROR   []<>a && []<>b && [](a -> <*>b) && [](

...

    0.022s  SAT     a U b
    0.014s  SAT     a W b
    0.014s  SAT     [](a -> <>b)
    0.008s  SAT     [](a -> X b)
    0.015s  UNSAT   []a && <>b
    0.007s  UNSAT   (a U b) && []!b
    0.007s  UNSAT   [](a -> <>b) && []<>a && []!b
    0.103s  UNSAT   [](a -> X(b U c)) && []<>a && []!c
    0.017s  SAT     <>(a && X b) && [](b -> X c)


    0.020s  SAT     []<>(a -> X(b U c))
    0.027s  SAT     (((a U b) U c) U d)
    0.058s  SAT     []<>a && []<>b && []<>c


    0.335s  SAT     []<>a && []<>b && []<>c && []<>d
    0.024s  SAT     <*>a
    0.018s  SAT     [*]a
    0.013s  UNSAT   Y a
    0.014s  SAT     Y a -> <*>a
    0.015s  SAT     a S b
    0.009s  UNSAT   <*>a && !a
    0.011s  UNSAT   [*]!a && <*>a
    0.013s  UNSAT   <*>(a && Y b)
    0.011s  UNSAT   Y Y Y Y a
    0.012s  SAT     <*><*><*>a
    0.017s  SAT     [](b -> <*>a)
    0.010s  SAT     [](b -> Y a)
    0.009s  UNSAT   [](b -> <*>a) && b
    0.020s  SAT     [](b -> (a S c))


    0.016s  SAT     [](a -> <*>(b && Y c))
    0.018s  SAT     []<>a && [](a -> <*>b)


    0.235s  SAT     []<>a && []<>b && [](a -> <*>b) && [](b -> <*>c)
--- pLTLpdTests done in 1.7s
129 measurements


## 6. Results

`results` is the raw table; the pivots below are what the comparison is for. If
pandas is not installed the same tables are printed as plain text.

In [6]:
results = [asdict(r) for r in rows]

try:
    import pandas as pd
    df = pd.DataFrame(results).drop(columns=["times"])
except ImportError:
    pd = None
    df = None
    print("pandas not available - using plain-text tables")


def cell(row_map, branch, key):
    row = row_map.get(branch)
    return None if row is None else row.get(key)


by_formula = {}
for r in results:
    by_formula.setdefault((r["group"], r["formula"], r["expected"]), {})[r["branch"]] = r


def text_table(header, lines):
    widths = [max(len(str(r[i])) for r in [header] + lines) for i in range(len(header))]
    sep = "  ".join("-" * w for w in widths)
    print("  ".join(str(h).ljust(w) for h, w in zip(header, widths)))
    print(sep)
    for line in lines:
        print("  ".join(str(c).ljust(w) for c, w in zip(line, widths)))


def show(title, header, lines):
    print(title)
    text_table(header, lines)
    print()

pandas not available - using plain-text tables


### 6.1 Verdict per branch

In [7]:
header = ["group", "formula", "expected"] + BRANCHES
lines = [[g, f, e or ""] + [cell(m, b, "verdict") or "-" for b in BRANCHES]
         for (g, f, e), m in by_formula.items()]

if pd is not None:
    verdicts = pd.DataFrame(lines, columns=header).set_index(["group", "formula"])
    display(verdicts)
else:
    show("verdict per branch", header, lines)

verdict per branch
group          formula                                           expected  masterTests  pLTLTests  pLTLpdTests
-------------  ------------------------------------------------  --------  -----------  ---------  -----------
propositional  a                                                 SAT       SAT          SAT        SAT        
propositional  a && !a                                           UNSAT     UNSAT        UNSAT      UNSAT      
propositional  a || !a                                           SAT       UNSAT        SAT        SAT        
tautology      a -> a                                            SAT       UNSAT        SAT        SAT        
tautology      []a -> a                                          SAT       SAT          SAT        SAT        
tautology      [](a -> b) -> ([]a -> []b)                        SAT       SAT          SAT        SAT        
contradiction  !(a -> a)                                         UNSAT     UNSAT        UNSAT

### 6.2 Time per branch (seconds, wall clock of the check)

In [8]:
header = ["group", "formula"] + BRANCHES
lines = []
for (g, f, e), m in by_formula.items():
    line = [g, f]
    for b in BRANCHES:
        s = cell(m, b, "seconds")
        line.append(round(s, 4) if s is not None else None)
    lines.append(line)

if pd is not None:
    timings = pd.DataFrame(lines, columns=header).set_index(["group", "formula"])
    display(timings.style.format("{:.4f}", na_rep="-")
            .background_gradient(axis=None, cmap="Blues"))
else:
    show("seconds per branch", header,
         [[c if not isinstance(c, float) else f"{c:.4f}" for c in line] for line in lines])

seconds per branch
group          formula                                           masterTests  pLTLTests  pLTLpdTests
-------------  ------------------------------------------------  -----------  ---------  -----------
propositional  a                                                 0.5817       0.6356     0.3641     
propositional  a && !a                                           0.0140       0.0130     0.0080     
propositional  a || !a                                           0.0227       0.0124     0.0146     
tautology      a -> a                                            0.0181       0.0166     0.0211     
tautology      []a -> a                                          0.0321       0.0175     0.0218     
tautology      [](a -> b) -> ([]a -> []b)                        0.0334       0.0219     0.0202     
contradiction  !(a -> a)                                         0.0252       0.0068     0.0103     
future         []a                                               0.0141 

### 6.3 Disagreements

Formulas whose verdict is not the same on every branch, or that do not match the
expectation written next to them. A verdict of `ERROR` on `masterTests` for a
past formula is expected (the operators do not exist there) and is listed for
completeness.

What this turns up as of this run: on `masterTests` a **contradiction compiles
to a 1-state property automaton with no ERROR state**, i.e. to "accept
everything". So the negation of a tautology looks satisfied by the universal
LTS and every tautology comes back `UNSAT` there (`a || !a`, `a -> a`, ...),
while the pLTL branches answer `SAT`. The pLTL branches assert the correct
behaviour in `TautologyDetectionTest`.

In [9]:
header = ["group", "formula", "expected"] + BRANCHES + ["detail"]
lines = []
for (g, f, e), m in by_formula.items():
    verdicts_here = [cell(m, b, "verdict") for b in BRANCHES]
    decided = [v for v in verdicts_here if v in ("SAT", "UNSAT", "VALID", "INVALID")]
    mismatch = len(set(decided)) > 1 or (e and any(v != e for v in decided))
    if mismatch:
        detail = next((cell(m, b, "detail") for b in BRANCHES
                       if cell(m, b, "verdict") in ("ERROR", "TIMEOUT")), "")
        lines.append([g, f, e or ""] + [v or "-" for v in verdicts_here] + [detail or ""])

if not lines:
    print("every branch agrees, and with the expected verdicts")
elif pd is not None:
    display(pd.DataFrame(lines, columns=header))
else:
    show("disagreements", header, lines)

disagreements
group          formula  expected  masterTests  pLTLTests  pLTLpdTests  detail
-------------  -------  --------  -----------  ---------  -----------  ------
propositional  a || !a  SAT       UNSAT        SAT        SAT                
tautology      a -> a   SAT       UNSAT        SAT        SAT                



### 6.4 Where the time goes (Java side) and how big the automaton is

In [10]:
header = ["group", "formula", "branch", "verdict", "compile_ms", "check_ms",
          "buchi_states", "buchi_transitions", "is_property"]
lines = [[r["group"], r["formula"], r["branch"], r["verdict"], r["compile_ms"],
          r["check_ms"], r["buchi_states"], r["buchi_transitions"], r["is_property"]]
         for r in results]

if pd is not None:
    display(df[["group", "formula", "branch", "verdict", "compile_ms", "check_ms",
                "buchi_states", "buchi_transitions", "is_property"]])
else:
    show("per-check detail", header, lines)

per-check detail
group          formula                                           branch       verdict  compile_ms  check_ms  buchi_states  buchi_transitions  is_property
-------------  ------------------------------------------------  -----------  -------  ----------  --------  ------------  -----------------  -----------
propositional  a                                                 masterTests  SAT      19          11        2             3                  True       
propositional  a && !a                                           masterTests  UNSAT    11          0         1             1                  True       
propositional  a || !a                                           masterTests  UNSAT    13          8         1             1                  True       
tautology      a -> a                                            masterTests  UNSAT    9           7         1             1                  True       
tautology      []a -> a                                    

### 6.5 Totals per branch

In [11]:
header = ["branch", "checks", "decided", "SAT", "UNSAT", "ERROR", "TIMEOUT",
          "total_s", "median_s", "max_s"]
lines = []
for b in BRANCHES:
    mine = [r for r in results if r["branch"] == b]
    secs = [r["seconds"] for r in mine if r["seconds"] is not None]
    counts = {k: sum(1 for r in mine if r["verdict"] == k)
              for k in ("SAT", "UNSAT", "ERROR", "TIMEOUT")}
    lines.append([
        b, len(mine), counts["SAT"] + counts["UNSAT"], counts["SAT"], counts["UNSAT"],
        counts["ERROR"], counts["TIMEOUT"],
        round(sum(secs), 3) if secs else None,
        round(statistics.median(secs), 4) if secs else None,
        round(max(secs), 4) if secs else None,
    ])

if pd is not None:
    display(pd.DataFrame(lines, columns=header).set_index("branch"))
else:
    show("totals per branch", header, lines)

totals per branch
branch       checks  decided  SAT  UNSAT  ERROR  TIMEOUT  total_s  median_s  max_s 
-----------  ------  -------  ---  -----  -----  -------  -------  --------  ------
masterTests  43      26       17   9      17     0        1.348    0.0181    0.5817
pLTLTests    43      43       30   13     0      0        2.697    0.0186    0.6356
pLTLpdTests  43      43       30   13     0      0        1.672    0.015     0.3641



## 7. Write the results / clean up

Two CSVs: the tidy one (a row per formula and branch, with every measurement)
and the by-formula one (a row per formula, verdict and seconds per branch). The
JSON dump next to the worktrees keeps the raw rows around to diff two runs.
Removing the
worktrees means the next run pays for a full `mvn clean compile` again, so it is
left commented out.

In [12]:
import csv

TIDY_COLUMNS = ["branch", "group", "formula", "expected", "verdict", "seconds",
                "compile_ms", "check_ms", "buchi_states", "buchi_transitions",
                "is_property", "alphabet", "detail"]

with open(CSV_RESULTS, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=TIDY_COLUMNS, extrasaction="ignore")
    writer.writeheader()
    for r in results:
        writer.writerow(r)
print("wrote", CSV_RESULTS, f"({len(results)} rows)")

wide_columns = ["group", "formula", "expected"]
for b in BRANCHES:
    wide_columns += [f"{b}:verdict", f"{b}:seconds"]

with open(CSV_BY_FORMULA, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(wide_columns)
    for (g, f, e), m in by_formula.items():
        row = [g, f, e or ""]
        for b in BRANCHES:
            seconds = cell(m, b, "seconds")
            row += [cell(m, b, "verdict") or "",
                    "" if seconds is None else f"{seconds:.4f}"]
        writer.writerow(row)
print("wrote", CSV_BY_FORMULA, f"({len(by_formula)} rows)")

out = os.path.join(WORKTREE_ROOT, "ltl_sat_runs.json")
with open(out, "w") as fh:
    json.dump({"mode": MODE, "alphabet": ALPHABET, "branches": BRANCHES,
               "results": results}, fh, indent=1)
print("wrote", out)

# for branch in BRANCHES:
#     if branch != CURRENT_BRANCH:
#         path = os.path.join(WORKTREE_ROOT, branch)
#         subprocess.run(["git", "-C", REPO_ROOT, "worktree", "remove", "--force", path])

wrote /home/lakso/Programas/mtsa/maven-root/mtsa/tools/ltl-sat/ltl_sat_results.csv (129 rows)
wrote /home/lakso/Programas/mtsa/maven-root/mtsa/tools/ltl-sat/ltl_sat_results_by_formula.csv (43 rows)
wrote /tmp/mtsa-ltlsat/ltl_sat_runs.json
